# RANKL M199K vs Wild-Type — Combined Computational Pipeline

MSBD 502, Computational Structural Biology — Sade Graves & LaPorchia Davis

Three methods, run on RANKL_WT.pdb and RANKL_M199K.pdb (PDB 3URF chain A, residues 162-317):

A. Structural comparison — py3Dmol + distance table around residue 199 (Lecture 3 approach, applied to the mutation site instead of a ligand).

B. PyRosetta ΔΔG — mutate residue 199 M→K on the WT pose, relax WT and mutant the same way (Lecture 6), compute ΔΔG_fold.

C. Molecular dynamics — OpenMM, AMBER force field (same family as Amber_2/Amber_3), RMSD/RMSF on WT vs mutant.

PDBs load from Google Drive, see the Mount Google Drive step below. Run top to bottom, no restart needed, everything installs with pip.


## 0. Setup

Installs via pip, no conda, no restart. PyRosetta install (second cell) takes a few minutes, let it finish.


In [ ]:
!pip install -q openmm pdbfixer MDAnalysis py3Dmol biopython matplotlib pandas numpy
print("Core packages installed.")


In [ ]:
!pip install -q pyrosetta-installer
import pyrosetta_installer
pyrosetta_installer.install_pyrosetta(skip_if_installed=True)
print("PyRosetta installed.")


### Mount Google Drive

Colab's disk resets on disconnect, so the PDBs and all outputs live in Drive instead.

One-time: make a Drive folder (e.g. `RANKL_M199K_project`), upload `RANKL_WT.pdb` and `RANKL_M199K.pdb` into it via drive.google.com. After that this cell just reconnects to where they already are.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_FOLDER = "/content/drive/MyDrive/RANKL_M199K_project"   # change this if you named your folder differently
import os
os.makedirs(DRIVE_FOLDER, exist_ok=True)
print("Drive mounted. Working folder:", DRIVE_FOLDER)


In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import py3Dmol
from Bio.PDB import PDBParser, NeighborSearch

import pyrosetta
from pyrosetta import pose_from_pdb, get_fa_scorefxn
from pyrosetta.toolbox import mutate_residue
from pyrosetta.rosetta.protocols.relax import FastRelax

# same init flags used in Lecture 6
pyrosetta.init("-ex1 -ex2 -use_input_sc -ignore_unrecognized_res -ignore_zero_occupancy false -mute all")

WT_PATH = f"{DRIVE_FOLDER}/RANKL_WT.pdb"
MUT_PATH = f"{DRIVE_FOLDER}/RANKL_M199K.pdb"
OUT_DIR = f"{DRIVE_FOLDER}/rankl_pipeline_output"
os.makedirs(OUT_DIR, exist_ok=True)

for p in (WT_PATH, MUT_PATH):
    assert os.path.exists(p), f"Can't find {p} -- upload RANKL_WT.pdb and RANKL_M199K.pdb into your {DRIVE_FOLDER} Drive folder first."
print("Found both structures. Ready. All outputs will be saved to Drive, not the temporary Colab disk.")


## A. Structural comparison (py3Dmol + distance table)

Same NeighborSearch approach as Lecture 3, applied to residue 199 instead of a ligand: find nearby residues, compare distances WT vs mutant.


In [ ]:
parser = PDBParser(QUIET=True)
struct_wt = parser.get_structure("WT", WT_PATH)
struct_mut = parser.get_structure("MUT", MUT_PATH)
model_wt = struct_wt[0]
model_mut = struct_mut[0]

chain_wt = next(iter(model_wt))
chain_mut = next(iter(model_mut))

res199_wt = chain_wt[199]
print("Wild-type residue 199:", res199_wt.get_resname())
print("Mutant residue 199:", chain_mut[199].get_resname())


In [ ]:
# find residues near 199 in the wild-type structure (same NeighborSearch approach as Lecture 3)
all_atoms_wt = list(model_wt.get_atoms())
ns_wt = NeighborSearch(all_atoms_wt)

site_atom = res199_wt["CA"]
nearby = ns_wt.search(site_atom.coord, 8.0)
neighbor_resis = sorted(set(a.get_parent().id[1] for a in nearby
                             if a.get_parent().id[1] != 199 and a.get_parent().id[0] == " "))
print(f"Residues within 8 A of residue 199 (CA-CA): {neighbor_resis}")

rows = []
for resi in neighbor_resis:
    try:
        d_wt = chain_wt[199]["CA"] - chain_wt[resi]["CA"]
        d_mut = chain_mut[199]["CA"] - chain_mut[resi]["CA"]
        rows.append({"residue": resi, "wt_distance_A": round(d_wt, 2), "mutant_distance_A": round(d_mut, 2),
                     "delta_A": round(d_mut - d_wt, 2)})
    except KeyError:
        continue

df_distances = pd.DataFrame(rows)
df_distances.to_csv(f"{OUT_DIR}/residue199_distance_table.csv", index=False)
df_distances


In [ ]:
# py3Dmol view: wild-type (cyan) vs mutant (orange) overlaid, zoomed on residue 199
with open(WT_PATH) as f:
    pdb_wt_text = f.read()
with open(MUT_PATH) as f:
    pdb_mut_text = f.read()

view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_wt_text, "pdb")
view.addModel(pdb_mut_text, "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "cyan"}})
view.setStyle({"model": 1}, {"cartoon": {"color": "orange"}})
view.addStyle({"model": 0, "resi": "199"}, {"stick": {"colorscheme": "cyanCarbon"}})
view.addStyle({"model": 1, "resi": "199"}, {"stick": {"colorscheme": "orangeCarbon"}})
view.zoomTo({"resi": "199"})
view.zoom(0.4)
view.show()


## B. PyRosetta ΔΔG (Lecture 6 approach)

Mutate residue 199 on the WT pose, relax WT and mutant the same way, compare total energy. ddG_fold > 0 = predicted destabilizing.


In [ ]:
wt_pose = pose_from_pdb(WT_PATH)
sfxn = get_fa_scorefxn()

info = wt_pose.pdb_info()
pose_199 = info.pdb2pose("A", 199)
print(f"PDB A:199 -> pose number {pose_199}  ({wt_pose.residue(pose_199).name3()})")
print(f"Starting REF2015 score: {sfxn(wt_pose):.1f} REU")


In [ ]:
def mutate_and_relax(pose, pose_res, new_aa, radius=8.0):
    '''Point mutation + repack neighbors + local minimization (same pattern as Lecture 6).'''
    mut = pose.clone()
    mutate_residue(mut, pose_res, new_aa, radius, sfxn)
    relax = FastRelax()
    relax.set_scorefxn(sfxn)
    relax.constrain_relax_to_start_coords(True)
    relax.coord_constrain_sidechains(True)
    relax.ramp_down_constraints(False)
    relax.max_iter(10)   # quick relax, raise this if you have more time/compute
    out = pose.clone()
    out = mut
    relax.apply(out)
    return out

N_REPEATS = 1   # Lecture 6 recommends 3-10 for production; raise this if time allows

wt_aa = wt_pose.residue(pose_199).name1()
E_wt = np.mean([sfxn(mutate_and_relax(wt_pose, pose_199, wt_aa)) for _ in range(N_REPEATS)])
E_mut = np.mean([sfxn(mutate_and_relax(wt_pose, pose_199, "K")) for _ in range(N_REPEATS)])

ddG_fold = E_mut - E_wt
print(f"E(wild-type, relaxed)  = {E_wt:8.1f} REU")
print(f"E(M199K, relaxed)      = {E_mut:8.1f} REU")
print(f"ddG_fold                = {ddG_fold:+8.1f} REU   (positive = destabilizing)")


In [ ]:
# per-residue energy around the mutation site, wild-type vs mutant, same style as Lecture 6's per-residue plot
mut_pose_final = mutate_and_relax(wt_pose, pose_199, "K")
wt_pose_final = mutate_and_relax(wt_pose, pose_199, wt_aa)

window = range(max(1, pose_199 - 10), min(wt_pose_final.total_residue(), pose_199 + 10) + 1)
res_E_wt = [wt_pose_final.energies().residue_total_energy(i) for i in window]
res_E_mut = [mut_pose_final.energies().residue_total_energy(i) for i in window]
pdb_numbers = [info.number(i) for i in window]

plt.figure(figsize=(10, 4))
x = np.arange(len(pdb_numbers))
plt.bar(x - 0.2, res_E_wt, width=0.4, label="Wild-type")
plt.bar(x + 0.2, res_E_mut, width=0.4, label="M199K")
plt.xticks(x, pdb_numbers, rotation=90)
plt.xlabel("Residue (PDB numbering)")
plt.ylabel("Per-residue REF2015 energy (REU)")
plt.title("Per-residue energy around the mutation site")
plt.axvline(x[pdb_numbers.index(199)], color="gray", linestyle="--")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/per_residue_energy.png", dpi=150)
plt.show()


## C. Molecular dynamics (OpenMM + AMBER force field)

Same engine/force field family as Amber_2/Amber_3, using OpenMM's built-in Amber14 force field directly, no conda/LEaP needed. Short run on both structures, then RMSD and per-residue RMSF.


In [ ]:
from pdbfixer import PDBFixer
from openmm.app import Modeller, PDBFile, ForceField, PME, HBonds, Simulation, DCDReporter, StateDataReporter
from openmm import LangevinMiddleIntegrator, Platform, unit
import sys

def prepare_system(pdb_path, out_prefix):
    fixer = PDBFixer(filename=pdb_path)
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.0)

    forcefield = ForceField("amber14-all.xml", "amber14/tip3pfb.xml")
    modeller = Modeller(fixer.topology, fixer.positions)
    modeller.addSolvent(forcefield, model="tip3p", padding=1.0 * unit.nanometers,
                         ionicStrength=0.15 * unit.molar)

    with open(f"{OUT_DIR}/{out_prefix}_solvated.pdb", "w") as f:
        PDBFile.writeFile(modeller.topology, modeller.positions, f)
    return forcefield, modeller

ff_wt, model_wt_md = prepare_system(WT_PATH, "wt")
ff_mut, model_mut_md = prepare_system(MUT_PATH, "mut")
print("Both structures solvated and ready for MD.")


In [ ]:
PRODUCTION_STEPS = 50000   # 100 ps at 2 fs/step -- short on purpose so this finishes on free Colab. Raise it if you have GPU time.
EQUILIBRATION_STEPS = 5000

def run_md(forcefield, modeller, out_prefix):
    system = forcefield.createSystem(modeller.topology, nonbondedMethod=PME,
                                      nonbondedCutoff=1.0 * unit.nanometers, constraints=HBonds)
    integrator = LangevinMiddleIntegrator(300 * unit.kelvin, 1 / unit.picosecond, 0.002 * unit.picoseconds)
    try:
        platform = Platform.getPlatformByName("CUDA")
    except Exception:
        platform = Platform.getPlatformByName("CPU")

    simulation = Simulation(modeller.topology, system, integrator, platform)
    simulation.context.setPositions(modeller.positions)

    print(f"[{out_prefix}] Minimizing...")
    simulation.minimizeEnergy()
    print(f"[{out_prefix}] Equilibrating...")
    simulation.context.setVelocitiesToTemperature(300 * unit.kelvin)
    simulation.step(EQUILIBRATION_STEPS)

    print(f"[{out_prefix}] Running production...")
    simulation.reporters.append(DCDReporter(f"{OUT_DIR}/{out_prefix}_traj.dcd", 500))
    simulation.reporters.append(StateDataReporter(sys.stdout, 5000, step=True, potentialEnergy=True, temperature=True))
    simulation.step(PRODUCTION_STEPS)

    with open(f"{OUT_DIR}/{out_prefix}_final.pdb", "w") as f:
        PDBFile.writeFile(simulation.topology, simulation.context.getState(getPositions=True).getPositions(), f)
    print(f"[{out_prefix}] Done.")

run_md(ff_wt, model_wt_md, "wt")
run_md(ff_mut, model_mut_md, "mut")


In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align

def compute_rmsd_rmsf(solvated_pdb, traj_dcd):
    u = mda.Universe(solvated_pdb, traj_dcd)
    protein = u.select_atoms("protein and backbone")
    R = rms.RMSD(protein, protein, ref_frame=0).run()
    rmsd_series = R.results.rmsd[:, 2]

    align.AlignTraj(u, u, select="protein and backbone", in_memory=True).run()
    ca = u.select_atoms("protein and name CA")
    R_f = rms.RMSF(ca).run()
    return rmsd_series, ca.resids, R_f.results.rmsf

rmsd_wt, resids_wt, rmsf_wt = compute_rmsd_rmsf(f"{OUT_DIR}/wt_solvated.pdb", f"{OUT_DIR}/wt_traj.dcd")
rmsd_mut, resids_mut, rmsf_mut = compute_rmsd_rmsf(f"{OUT_DIR}/mut_solvated.pdb", f"{OUT_DIR}/mut_traj.dcd")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(rmsd_wt, label="Wild-type")
axes[0].plot(rmsd_mut, label="M199K")
axes[0].set_xlabel("Frame"); axes[0].set_ylabel("Backbone RMSD (A)")
axes[0].set_title("RMSD over the simulation"); axes[0].legend()

axes[1].plot(resids_wt, rmsf_wt, label="Wild-type")
axes[1].plot(resids_mut, rmsf_mut, label="M199K")
axes[1].axvline(199, color="gray", linestyle="--", label="Residue 199")
axes[1].set_xlabel("Residue number"); axes[1].set_ylabel("RMSF (A)")
axes[1].set_title("Per-residue flexibility"); axes[1].legend()

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/rmsd_rmsf_comparison.png", dpi=150)
plt.show()

print("Mean RMSD -- wild-type: %.2f A, M199K: %.2f A" % (rmsd_wt.mean(), rmsd_mut.mean()))
idx_wt = list(resids_wt).index(199) if 199 in resids_wt else None
idx_mut = list(resids_mut).index(199) if 199 in resids_mut else None
if idx_wt is not None and idx_mut is not None:
    print("RMSF at residue 199 -- wild-type: %.2f A, M199K: %.2f A" % (rmsf_wt[idx_wt], rmsf_mut[idx_mut]))


## Summary

- A: distance table, how local geometry around 199 shifts WT vs mutant.
- B: ddG_fold, positive = destabilizing.
- C: RMSD = overall drift over time, RMSF = where it's most flexible, watch for a spike near 199.

Limitations: N_REPEATS = 1 for ddG_fold (Lecture 6 suggests 3-10 for production, a single run is noisy, +/- 0.5-1 REU). PRODUCTION_STEPS kept short (100 ps) to fit a free Colab runtime. Raise either if there's more compute time before the deadline.
